# Data preparation — 12-week and 17-week STREAM inputs

Uses `adata.X` and available `Age` / `cell types` annotations. Sections 1–3 are a **single cell** that checks for prior QC/normalization and avoids processing twice. Files save to `../Results/Data_prep_outputs`.


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

SOURCE = Path('../Data/HTA08.v01.A06.Science_human_tcells.h5ad')
OUTPUT_DIR = Path('../Results/Data_prep_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_UMI = 2000
MIN_GENES = 500
MAX_GENES = 7000
TARGET_SUM = 10000

adata = sc.read_h5ad(SOURCE)
print('Dataset:', adata.shape)
print('Age:', adata.obs['Age'].value_counts(dropna=False).to_dict())


Dataset: (76994, 33694)
Age: {'16w': 9577, '17w': 7673, '14w': 6548, '12w': 6116, '11w': 5615, '13w': 5555, '30m': 5286, '15m': 4957, '3m': 4591, '6m': 4505, '13y': 3976, '8w': 2638, '10w': 2607, '7w': 2222, 'nan': 1981, '9w': 1963, '10m': 1184}


/home/eren/miniforge3/envs/scanpy-env/lib/python3.11/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [5]:
adata.obs['cell types'].cat.categories.tolist()

['CD4+T',
 'CD8+T',
 'CD8αα(I)',
 'CD8αα(II)',
 'DN(P)',
 'DN(Q)',
 'DN(early)',
 'DP(P)',
 'DP(Q)',
 'NKT',
 'T(agonist)',
 'Th17',
 'Treg',
 'Treg(diff)',
 'αβT(entry)',
 'γδT']

## 1–3. Check once, filter only if needed, normalize only if needed

This single block examines `adata.X` and the observed cell-level metrics once. Integer-valued `.X` is treated as UMI counts; for those data it verifies or applies **≥2,000 UMIs**, **≥500 genes**, and **≤7,000 genes**. If the matrix is already noninteger/normalized, no second normalization is performed. The gene-based QC bounds can still be checked, but the historical raw-UMI threshold **cannot** be established from normalized expression; that limitation is recorded in the report.


In [6]:
# One-time input inspection, QC, age split and conditional normalization.
adata.var_names_make_unique()
adata.obs_names_make_unique()

X = adata.X
values = X.data if sparse.issparse(X) else np.asarray(X).ravel()
if not np.isfinite(values).all() or (values < 0).any():
    raise ValueError('adata.X contains negative or non-finite expression values.')
is_raw_counts = bool(np.allclose(values, np.rint(values), rtol=0, atol=1e-6))

n_genes = (np.asarray((X > 0).sum(axis=1)).ravel()
           if sparse.issparse(X) else np.count_nonzero(X > 0, axis=1))
cell_totals = np.asarray(X.sum(axis=1)).ravel()

def normalize_age(value):
    s = str(value).strip().lower()
    if re.fullmatch(r'12\s*[-_ ]?\s*(?:w|wk|wks|week|weeks)', s):
        return '12w'
    if re.fullmatch(r'17\s*[-_ ]?\s*(?:w|wk|wks|week|weeks)', s):
        return '17w'
    return None

adata.obs['age_group'] = adata.obs['Age'].map(normalize_age)
adata.obs['n_genes_detected'] = n_genes
adata.obs['expression_sum'] = cell_totals
adata.obs['pass_gene_qc'] = (n_genes >= MIN_GENES) & (n_genes <= MAX_GENES)
if is_raw_counts:
    adata.obs['n_umi_raw'] = cell_totals
    adata.obs['pass_umi_qc'] = cell_totals >= MIN_UMI
    adata.obs['pass_qc'] = adata.obs['pass_gene_qc'] & adata.obs['pass_umi_qc']
else:
    adata.obs['pass_umi_qc'] = pd.Series(
    'not_verifiable',
    index=adata.obs.index,
    dtype='string')
    adata.obs['pass_qc'] = adata.obs['pass_gene_qc']
    print('NOTE: X is noninteger (possibly normalized). Raw 2,000-UMI QC cannot be verified or reapplied.')

print('Input:', 'raw integer counts' if is_raw_counts else 'noninteger expression (treated as preprocessed)')
print('Gene QC:', 'already satisfied' if adata.obs['pass_gene_qc'].all() else 'filtering failing cells')
if is_raw_counts:
    print('UMI QC:', 'already satisfied' if adata.obs['pass_umi_qc'].all() else 'filtering failing cells')

processed = {}
report = []
for age in ['12w', '17w']:
    subset = adata.obs['age_group'].eq(age)
    keep = subset & adata.obs['pass_qc']
    if not subset.any():
        raise ValueError(f'No cells labelled {age}; inspect adata.obs["Age"].')
    if not keep.any():
        raise ValueError(f'No {age} cells remain after QC.')
    sample = adata[keep.to_numpy(), :].copy()
    if is_raw_counts:
        # Normalization is applied only to a raw integer count matrix.
        sc.pp.normalize_total(sample, target_sum=TARGET_SUM)
        sc.pp.log1p(sample)
        normalization = 'normalize_total(10000) + log1p'
    else:
        normalization = 'unchanged: input already noninteger/preprocessed'
    sample.uns['data_prep'] = {
        'input_type': 'raw integer counts' if is_raw_counts else 'noninteger/preprocessed',
        'normalization': normalization,
        'raw_umi_qc_verified': bool(is_raw_counts),
        'raw_umi_min': MIN_UMI,
        'min_genes': MIN_GENES,
        'max_genes': MAX_GENES,
        'age': age,
    }
    sample.write_h5ad(OUTPUT_DIR / f'{age}_filtered_normalized.h5ad', compression='gzip')
    processed[age] = sample
    report.append({
        'age': age,
        'cells_before': int(subset.sum()),
        'cells_after': int(keep.sum()),
        'removed_by_gene_qc': int((subset & ~adata.obs['pass_gene_qc']).sum()),
        'removed_by_umi_qc': int((subset & ~adata.obs['pass_umi_qc']).sum()) if is_raw_counts else 'unknown: no raw UMI counts',
        'raw_umi_qc_verified': is_raw_counts,
        'normalization': normalization,
    })
    print(age, sample.shape, normalization)

pd.DataFrame(report).to_csv(OUTPUT_DIR / 'cell_qc_summary.csv', index=False)
adata.obs.to_csv(OUTPUT_DIR / 'all_cells_qc_metrics.csv', index_label='cell_id')
print(pd.DataFrame(report).to_string(index=False))


NOTE: X is noninteger (possibly normalized). Raw 2,000-UMI QC cannot be verified or reapplied.
Input: noninteger expression (treated as preprocessed)
Gene QC: already satisfied
12w (6116, 33694) unchanged: input already noninteger/preprocessed
17w (7673, 33694) unchanged: input already noninteger/preprocessed
age  cells_before  cells_after  removed_by_gene_qc          removed_by_umi_qc  raw_umi_qc_verified                                    normalization
12w          6116         6116                   0 unknown: no raw UMI counts                False unchanged: input already noninteger/preprocessed
17w          7673         7673                   0 unknown: no raw UMI counts                False unchanged: input already noninteger/preprocessed


## 4. Export STREAM input trio per time point

STREAM `st.read()` transposes the tab-delimited input on import. Therefore **TSV rows must be genes and columns must be cells**, with gene names in the first column and cell IDs in the header. `st.add_cell_labels()` reads a **headerless single-column label TSV**, in precisely the same cell order as the matrix. `st.add_cell_colors()` reads a **headerless two-column label/color TSV**; its first column contains label names and its second contains hex colors.

Labels come from a cell-type annotation when available. Otherwise, every cell is labelled by its week so there is no inferred cell-type assignment.

In [7]:
from matplotlib import colors as mcolors

# Edit this list if the relevant metadata field has a different name.
LABEL_CANDIDATES = ['cell types', 'cell_type', 'celltype', 'CellType', 'Cell_type',
                    'cell_type_annot', 'cell_type_annotation', 'annotation',
                    'cluster', 'leiden', 'louvain']
label_column = next((c for c in LABEL_CANDIDATES if c in adata.obs.columns), None)
print('STREAM label source:', label_column if label_column else 'Age (fallback)')

def save_stream_input(sample, age):
    expression_path = OUTPUT_DIR / f'{age}_expression.tsv.gz'
    labels_path = OUTPUT_DIR / f'{age}_cell_label.tsv.gz'
    colors_path = OUTPUT_DIR / f'{age}_cell_label_color.tsv.gz'

    # Sparse-friendly export in blocks of genes: columns are cells; rows genes.
    gene_by_cell = sample.X.T.tocsr() if sparse.issparse(sample.X) else sample.X.T
    genes = sample.var_names.astype(str)
    cells = sample.obs_names.astype(str)
    for start in range(0, sample.n_vars, 256):
        end = min(start + 256, sample.n_vars)
        block = gene_by_cell[start:end]
        block = block.toarray() if sparse.issparse(block) else np.asarray(block)
        frame = pd.DataFrame(block, index=genes[start:end], columns=cells)
        frame.to_csv(expression_path, sep='\t', compression='gzip',
                     mode='wt' if start == 0 else 'at',
                     header=(start == 0), index=True,
                     index_label=None if start else 'gene')

    if label_column is None:
        labels = pd.Series(age, index=sample.obs_names, dtype=str)
    else:
        labels = sample.obs[label_column].astype('string').fillna('unknown').astype(str)
    labels = labels.str.replace('/', '-', regex=False).reset_index(drop=True)
    labels.to_csv(labels_path, sep='\t', header=False, index=False, compression='gzip')

    unique_labels = sorted(labels.unique())
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap('tab20', max(1, len(unique_labels)))
    color_table = pd.DataFrame({
        'label': unique_labels,
        'color': [mcolors.to_hex(cmap(i)) for i in range(len(unique_labels))]
    })
    color_table.to_csv(colors_path, sep='\t', header=False, index=False, compression='gzip')

    # Check exactly what STREAM will read.
    header = pd.read_csv(expression_path, sep='\t', nrows=0).columns.tolist()
    assert header[1:] == list(cells), 'Expression matrix cell order differs'
    assert len(labels) == sample.n_obs
    assert set(labels) <= set(color_table['label'])
    print(f'{age}: {sample.n_obs} cells, {sample.n_vars} genes, {len(unique_labels)} labels')
    return {'age': age, 'expression': str(expression_path), 'cell_labels': str(labels_path),
            'label_colors': str(colors_path), 'cells': sample.n_obs, 'genes': sample.n_vars}

manifest = pd.DataFrame([save_stream_input(processed[age], age) for age in ['12w', '17w']])
manifest.to_csv(OUTPUT_DIR / 'stream_input_manifest.csv', index=False)
print(manifest.to_string(index=False))


STREAM label source: cell types
12w: 6116 cells, 33694 genes, 16 labels
17w: 7673 cells, 33694 genes, 16 labels
age                                         expression                                        cell_labels                                             label_colors  cells  genes
12w ../Results/Data_prep_outputs/12w_expression.tsv.gz ../Results/Data_prep_outputs/12w_cell_label.tsv.gz ../Results/Data_prep_outputs/12w_cell_label_color.tsv.gz   6116  33694
17w ../Results/Data_prep_outputs/17w_expression.tsv.gz ../Results/Data_prep_outputs/17w_cell_label.tsv.gz ../Results/Data_prep_outputs/17w_cell_label_color.tsv.gz   7673  33694


## 5. Files and STREAM usage

This produces `12w_expression.tsv.gz` / `17w_expression.tsv.gz`, matching `*_cell_label.tsv.gz` and `*_cell_label_color.tsv.gz`, two normalized `.h5ad` files (with original counts retained as a layer), and a QC summary/metrics file and manifest — all in `../Results/Data_prep_outputs/`.

In [8]:
for age in ['12w', '17w']:
    print(f"\n# STREAM inputs for {age}:")
    print(f"adata = st.read(file_name='{OUTPUT_DIR / (age + '_expression.tsv.gz')}', "
          f"workdir='../Results/STREAM_outputs_{age}')")
    print(f"st.add_cell_labels(adata, file_name='{OUTPUT_DIR / (age + '_cell_label.tsv.gz')}')")
    print(f"st.add_cell_colors(adata, file_name='{OUTPUT_DIR / (age + '_cell_label_color.tsv.gz')}')")



# STREAM inputs for 12w:
adata = st.read(file_name='../Results/Data_prep_outputs/12w_expression.tsv.gz', workdir='../Results/STREAM_outputs_12w')
st.add_cell_labels(adata, file_name='../Results/Data_prep_outputs/12w_cell_label.tsv.gz')
st.add_cell_colors(adata, file_name='../Results/Data_prep_outputs/12w_cell_label_color.tsv.gz')

# STREAM inputs for 17w:
adata = st.read(file_name='../Results/Data_prep_outputs/17w_expression.tsv.gz', workdir='../Results/STREAM_outputs_17w')
st.add_cell_labels(adata, file_name='../Results/Data_prep_outputs/17w_cell_label.tsv.gz')
st.add_cell_colors(adata, file_name='../Results/Data_prep_outputs/17w_cell_label_color.tsv.gz')
